# Notebook 14 — Fannie Mae data compatibility and target validation
## Stage 1B — Schema alignment
The earlier inspection removed the first and last empty tokens. The official layout has a Reference Pool ID at position 1, not applicable to Single-Family Loan Performance. Removing the leading empty token may therefore have shifted the mapping. Keep every token and test this hypothesis before reading full loan histories.

Reference: [Fannie Mae glossary, July 29, 2024](https://capitalmarkets.fanniemae.com/sites/capmrkt/files/2024-07/crt-file-layout-and-glossary.pdf). [2026 disclosure changes](https://capitalmarkets.fanniemae.com/credit-risk-transfer/single-family-credit-risk-transfer/enhance-credit-risk-transfer-disclosures). The local download vintage is not yet known. This audit checks key positions and profiles every physical token, but does not certify the full release-specific schema or construct targets.

## 14.1 Locate the existing file
Use the same source path as the earlier inspection. No raw files will be changed.

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from itertools import islice
import csv, json, re, zipfile

ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
SOURCE = ROOT / 'data/raw/Fannie_Mae/2018Q1/2018Q1.csv'
assert SOURCE.is_file(), f'File missing: {SOURCE}'
SIZE = SOURCE.stat().st_size
print('File:', SOURCE)
print('Bytes:', SIZE)

File: e:\capstone_project\mortgage-model-reliability\data\raw\Fannie_Mae\2018Q1\2018Q1.csv
Bytes: 6928164686


## 14.2 Inspect the physical positions
Read up to 200 records at each of seven offsets. Preserve leading and trailing empty fields. This is a diagnostic sample of monthly rows, not a random loan sample.

In [2]:
rows = []
widths = Counter()
for fraction in (0, .1, .3, .5, .7, .9, .99):
    with SOURCE.open('rb') as f:
        f.seek(int(SIZE * fraction))
        if fraction:
            f.readline()
        count = 0
        for raw in islice(f, 200):
            text = raw.decode('utf-8-sig').rstrip('\r\n')
            row = next(csv.reader([text], delimiter='|'))
            rows.append(row)
            widths[len(row)] += 1
            count += 1
        print(f'Offset {fraction:.0%}: {count} rows')
assert rows, 'No records found'
print('Physical token counts:', dict(widths))
print('Leading blank:', sum(r[0] == '' for r in rows))
print('Trailing blank:', sum(r[-1] == '' for r in rows))
assert len(widths) == 1, 'Inconsistent record widths; stop and inspect.'
WIDTH = len(rows[0])

Offset 0%: 200 rows
Offset 10%: 200 rows
Offset 30%: 200 rows
Offset 50%: 200 rows
Offset 70%: 200 rows
Offset 90%: 200 rows
Offset 99%: 200 rows
Physical token counts: {113: 1400}
Leading blank: 1400
Trailing blank: 1400


## 14.3 Define documented anchor positions
Use field positions from the 2024 glossary, not indices from the earlier stripped rows. The date, range and code tests below are diagnostic checks; they are not cleaning rules or final eligibility rules. Blank optional values are tracked separately.

In [3]:
def month(v):
    return bool(re.fullmatch(r'(0[1-9]|1[0-2])(19|20)\d{2}', v))

def number(v, low, high):
    try:
        return low <= float(v) <= high
    except ValueError:
        return False

# position: (field, test, required anchor)
FIELDS = {
    2: ('loan_id', lambda v: bool(re.fullmatch(r'[A-Za-z0-9]{1,12}', v)) and not month(v), True),
    3: ('report_month', month, True),
    4: ('channel', lambda v: v in {'R','C','B'}, True),
    8: ('original_rate', lambda v: number(v, .01, 30), True),
    10: ('original_upb', lambda v: number(v, 1, 100000000), True),
    13: ('original_term', lambda v: v.isdigit() and number(v, 1, 600), True),
    14: ('origination_month', month, True),
    15: ('first_payment_month', month, True),
    16: ('loan_age', lambda v: bool(re.fullmatch(r'-?\d{1,3}', v)), False),
    19: ('maturity_month', month, False),
    20: ('ltv', lambda v: number(v, 1, 300), False),
    21: ('cltv', lambda v: number(v, 1, 300), False),
    22: ('borrowers', lambda v: v.isdigit() and number(v, 1, 99), True),
    23: ('dti', lambda v: number(v, 0, 99), False),
    24: ('borrower_fico', lambda v: v.isdigit() and number(v, 300, 850), False),
    25: ('coborrower_fico', lambda v: v.isdigit() and number(v, 300, 850), False),
    26: ('first_time_homebuyer', lambda v: v in {'Y','N'}, False),
    27: ('purpose', lambda v: v in {'C','R','P','U'}, True),
    28: ('property_type', lambda v: v in {'CO','CP','PU','MH','SF'}, True),
    29: ('units', lambda v: v in {'1','2','3','4'}, True),
    30: ('occupancy', lambda v: v in {'P','S','I','U'}, True),
    31: ('state', lambda v: bool(re.fullmatch(r'[A-Z]{2}', v)), True),
    32: ('msa', lambda v: bool(re.fullmatch(r'\d{5}', v)), False),
    33: ('zip_short', lambda v: bool(re.fullmatch(r'\d{3}', v)), False),
    34: ('mi_percent', lambda v: number(v, 0, 100), False),
    35: ('amortization_type', lambda v: v in {'FRM','ARM'}, True),
    37: ('interest_only', lambda v: v in {'Y','N'}, False),
    40: ('delinquency', lambda v: v == 'XX' or bool(re.fullmatch(r'\d{1,2}', v)), False),
    41: ('payment_history', lambda v: len(v) <= 48 and len(v) % 2 == 0 and bool(re.fullmatch(r'(?:\d{2}|XX)+', v)), False),
    42: ('modification', lambda v: v in {'Y','N'}, False),
    44: ('zero_balance_code', lambda v: v.isdigit() and len(v) <= 3, False),
    45: ('zero_balance_month', month, False),
    51: ('last_paid_installment', month, False),
    52: ('foreclosure_month', month, False),
    53: ('disposition_month', month, False),
}
print('Anchor fields:', len(FIELDS))

Anchor fields: 35


## 14.4 Compare indexing hypotheses
Offset 0 means glossary position p is Python index p-1. Offset +1 reproduces the earlier leading-token removal. Offset -1 tests the alternative. A blank optional field is not evidence of correct alignment. Do not automatically select a layout based on scores.

In [4]:
audit = []
for offset in (-1, 0, 1):
    for pos, (name, check, required) in FIELDS.items():
        idx = pos - 1 + offset
        assert 0 <= idx < WIDTH
        vals = [r[idx].strip() for r in rows]
        present = [v for v in vals if v]
        valid = sum(check(v) for v in present)
        audit.append(dict(offset=offset, glossary_position=pos, physical_position=idx+1,
                          field=name, required_anchor=required, rows=len(vals),
                          blank=len(vals)-len(present), nonblank=len(present),
                          valid_nonblank=valid, invalid_nonblank=len(present)-valid,
                          valid_fraction_nonblank=round(valid/len(present), 5) if present else None))
summary = []
for offset in (-1, 0, 1):
    anchors = [a for a in audit if a['offset']==offset and a['required_anchor']]
    passed = sum(a['valid_nonblank'] / a['rows'] >= .99 for a in anchors)
    summary.append(dict(offset=offset, anchors_passing=passed, anchors_total=len(anchors)))
    print(summary[-1])
print('Offset 0 detail:')
for a in audit:
    if a['offset']==0:
        print(a['field'], 'present:', a['nonblank'], 'invalid:', a['invalid_nonblank'])

{'offset': -1, 'anchors_passing': 1, 'anchors_total': 15}
{'offset': 0, 'anchors_passing': 15, 'anchors_total': 15}
{'offset': 1, 'anchors_passing': 2, 'anchors_total': 15}
Offset 0 detail:
loan_id present: 1400 invalid: 0
report_month present: 1400 invalid: 0
channel present: 1400 invalid: 0
original_rate present: 1400 invalid: 0
original_upb present: 1400 invalid: 0
original_term present: 1400 invalid: 0
origination_month present: 1400 invalid: 0
first_payment_month present: 1400 invalid: 0
loan_age present: 1376 invalid: 0
maturity_month present: 1376 invalid: 0
ltv present: 1400 invalid: 0
cltv present: 1400 invalid: 0
borrowers present: 1400 invalid: 0
dti present: 1400 invalid: 0
borrower_fico present: 1395 invalid: 0
coborrower_fico present: 515 invalid: 0
first_time_homebuyer present: 1400 invalid: 0
purpose present: 1400 invalid: 0
property_type present: 1400 invalid: 0
units present: 1400 invalid: 0
occupancy present: 1400 invalid: 0
state present: 1400 invalid: 0
msa present

## 14.5 Profile every physical field
Export only aggregate token characteristics, not loan identifiers, individual rows or raw values. Positions beyond the documented anchors remain unverified, including trailing empty fields and any release-specific additions.

In [5]:
profile = []
for j in range(WIDTH):
    vals = [r[j].strip() for r in rows]
    nonblank = [v for v in vals if v]
    profile.append(dict(physical_position=j+1,
                        candidate_name=FIELDS.get(j+1, ('unverified',))[0],
                        rows=len(vals), blank=len(vals)-len(nonblank),
                        numeric_tokens=sum(bool(re.fullmatch(r'-?\d+(?:\.\d+)?', v)) for v in nonblank),
                        month_tokens=sum(month(v) for v in nonblank),
                        min_length=min(map(len, nonblank), default=0),
                        max_length=max(map(len, nonblank), default=0)))
print('Profiled physical positions:', len(profile))
print('Fully blank sampled positions:', [p['physical_position'] for p in profile if p['blank']==len(rows)])

Profiled physical positions: 113
Fully blank sampled positions: [1, 7, 11, 38, 39, 43, 47, 48, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 67, 68, 69, 70, 71, 72, 75, 76, 77, 78, 82, 83, 84, 85, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 104, 107, 108, 110, 112, 113]


## 14.6 Save the review evidence
The output is a small aggregate ZIP. Review the alignment and release vintage before approving a schema. Do not infer terminal event equivalence or build the target in this stage.

In [6]:
OUT = ROOT / 'data/processed/Fannie_Mae/notebook_14_compatibility' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT.mkdir(parents=True, exist_ok=False)
for name, records in [('anchor_checks.csv', audit), ('offset_comparison.csv', summary), ('physical_position_profile.csv', profile)]:
    with (OUT/name).open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)
metadata = dict(source_name=SOURCE.name, file_bytes=SIZE, sampled_rows=len(rows),
                physical_width=WIDTH, width_counts=dict(widths),
                leading_blank=sum(r[0]=='' for r in rows), trailing_blank=sum(r[-1]=='' for r in rows),
                glossary_version='2024-07-29', local_release_vintage='unverified',
                status='pending_manual_review', target_constructed=False, models_scored=False)
(OUT/'audit_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
ZIP = OUT/'notebook_14_stage1b_review.zip'
with zipfile.ZipFile(ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p != ZIP:
            z.write(p, p.name)
print('Review ZIP:', ZIP)

Review ZIP: e:\capstone_project\mortgage-model-reliability\data\processed\Fannie_Mae\notebook_14_compatibility\20261124T143322777866Z\notebook_14_stage1b_review.zip


## Review gate
Share the ZIP. If the layout is ambiguous or release-specific fields are unresolved, obtain the glossary supplied with the local download before proceeding. No Freddie artifacts or Git files were modified by this audit.